# 📘 Módulo 00: Fundamentos del Modelo Relacional & Álgebra Relacional
### 🎓 Fundamentación Rigurosa basada en Teoría de Conjuntos, Notación Matemática y Código SQL
**Asignatura:** Bases de Datos Avanzadas  
**Entorno:** Google Colab / DuckDB (100% In-Memory)  
---
## 🎯 Objetivos de Aprendizaje
1. Comprender la base matemática del **Modelo Relacional** mediante la **Teoría de Conjuntos**.
2. Representar relaciones como subconjuntos del producto cartesiano de dominios: $R \subseteq D_1 \times D_2 \times \dots \times D_n$.
3. Dominar los operadores algebraicos unarios (Selección $\sigma$, Proyección $\pi$) y binarios (Unión $\cup$, Intersección $\cap$, Diferencia $-$, Producto Cartesiano $\times$, Join $\bowtie$).
4. Visualizar el comportamiento interno del motor al evaluar expresiones algebraicas complejas.
5. Traducir notación matemática formal a consultas SQL ejecutables en **DuckDB**.

In [ ]:
# ==============================================================================
# 🚀 SETUP INICIAL AUTO-CONTENIDO: DUCKDB & GOOGLE COLAB - MÓDULO 00
# ==============================================================================
!pip install duckdb duckdb-engine jupysql -q

import duckdb

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# 1. Conectar a motor DuckDB en memoria
conn = duckdb.connect(database=':memory:')

# 2. Poblar Esquema de Referencia Empresa CUN (100% In-Memory / Zero 404 Error)
conn.execute("""
CREATE TABLE IF NOT EXISTS departamentos (
    depto_id INT PRIMARY KEY,
    nombre VARCHAR,
    presupuesto DECIMAL(15,2),
    ciudad VARCHAR
);
INSERT INTO departamentos VALUES
(1, 'Tecnología e Innovación', 150000000.00, 'Bogotá'),
(2, 'Ventas y Comercio', 200000000.00, 'Medellín'),
(3, 'Operaciones y Logística', 120000000.00, 'Cali'),
(4, 'Recursos Humanos', 80000000.00, 'Bogotá'),
(5, 'Finanzas y Contabilidad', 110000000.00, 'Barranquilla');

CREATE TABLE IF NOT EXISTS empleados (
    empleado_id INT PRIMARY KEY,
    nombre VARCHAR,
    salario DECIMAL(15,2),
    fecha_ingreso DATE,
    jefe_id INT,
    depto_id INT
);
INSERT INTO empleados VALUES
(101, 'Carlos Mendoza', 8500000.00, '2021-03-15', NULL, 1),
(102, 'Ana María Gómez', 6200000.00, '2022-01-10', 101, 1),
(103, 'Luis Fernando Torres', 4800000.00, '2023-05-20', 101, 1),
(104, 'Sofía Ramírez', 9000000.00, '2020-11-01', NULL, 2),
(105, 'Diego Alejandro Silva', 5500000.00, '2022-08-14', 104, 2),
(106, 'Valentina Morales', 5100000.00, '2023-02-28', 104, 2),
(107, 'Javier Castro', 7000000.00, '2019-06-12', NULL, 3),
(108, 'Mariana López', 4200000.00, '2024-01-15', 107, 3),
(109, 'Andrés Felipe Herrera', 6500000.00, '2021-09-01', NULL, 4),
(110, 'Camila Ospina', 5800000.00, '2022-10-05', NULL, 5);

CREATE TABLE IF NOT EXISTS clientes (
    cliente_id INT PRIMARY KEY,
    nombre VARCHAR,
    ciudad VARCHAR,
    categoria_cliente VARCHAR,
    fecha_registro DATE
);
INSERT INTO clientes VALUES
(201, 'Tech Solutions S.A.S.', 'Bogotá', 'Empresarial', '2022-04-10'),
(202, 'Comercializadora del Valle', 'Cali', 'Empresarial', '2021-08-19'),
(203, 'Juan Pablo Restrepo', 'Medellín', 'Individual', '2023-01-12'),
(204, 'Inversiones Caribe Ltd.', 'Barranquilla', 'Empresarial', '2022-11-30'),
(205, 'Diana Marcela Vargas', 'Bogotá', 'VIP', '2020-05-14'),
(206, 'Servicios Logísticos del Eje', 'Pereira', 'Empresarial', '2023-07-22'),
(207, 'Mateo Gutiérrez', 'Bucaramanga', 'Individual', '2024-02-01');
""")

%sql conn
print('✅ Entorno DuckDB inicializado para el Módulo 00 (Álgebra Relacional).')

## 1. Notación Formal del Álgebra Relacional
El **Álgebra Relacional** es un lenguaje de consulta procedimental introducido por E. F. Codd. Toma una o dos relaciones como entrada y produce una nueva relación como resultado.
### 📐 Matriz de Operadores Algebraicos vs SQL:
| Operación | Símbolo | Notación Formal | Equivalencia SQL |
| :--- | :---: | :--- | :--- |
| **Selección** | $\sigma$ | $\sigma_{\text{predicado}}(R)$ | `SELECT * FROM R WHERE predicado` |
| **Proyección** | $\pi$ | $\pi_{A_1, A_2}(R)$ | `SELECT A1, A2 FROM R` |
| **Unión** | $\cup$ | $R \cup S$ | `SELECT * FROM R UNION SELECT * FROM S` |
| **Intersección** | $\cap$ | $R \cap S$ | `SELECT * FROM R INTERSECT SELECT * FROM S` |
| **Diferencia** | $-$ / $\setminus$ | $R - S$ | `SELECT * FROM R EXCEPT SELECT * FROM S` |
| **Producto Cartesiano** | $\times$ | $R \times S$ | `SELECT * FROM R CROSS JOIN S` |
| **Renombramiento** | $\rho$ | $\rho_{S}(R)$ | `SELECT * FROM R AS S` |
| **Natural Join** | $\bowtie$ | $R \bowtie_{\text{condición}} S$ | `SELECT * FROM R JOIN S ON ...` |

## 2. Operadores Unarios: Selección ($\sigma$) y Proyección ($\pi$)
- **Selección ($\sigma$):** Filtra **filas** (horizontales) que cumplen un predicado booleano.
- **Proyección ($\pi$):** Selecciona **columnas** (verticales) específicas y elimina duplicados resultado.
### 💡 Composición de Operaciones:
Podemos anidar operaciones:  
$$\pi_{\text{nombre}, \text{salario}}(\sigma_{\text{depto\_id} = 1 \land \text{salario} > 6000000}(\text{empleados}))$$

In [ ]:
%%sql
-- 1. Selección (σ): Empleados del depto 1 con salario > 6.000.000
-- Expresión: σ_{depto_id = 1 ∧ salario > 6000000}(empleados)
SELECT *
FROM empleados
WHERE depto_id = 1 AND salario > 6000000;

-- 2. Proyección (π) combinada con Selección (σ):
-- Expresión: π_{nombre, salario}(σ_{depto_id = 1}(empleados))
SELECT nombre, salario
FROM empleados
WHERE depto_id = 1;

## 3. Operadores de Conjuntos: Unión ($\cup$), Intersección ($\cap$) y Diferencia ($-$)
Para aplicar operadores de conjuntos, las relaciones deben ser **Compatibles de Unión**:
1. Deben tener la **misma aridad** (mismo número de columnas).
2. Los **dominios (tipos de datos)** de las columnas correspondientes deben ser compatibles.
```text
  ┌──────────────────────┐             ┌──────────────────────┐
  │  Ciudades Clientes   │   UNION     │  Ciudades Deptos     │
  │  (Bogotá, Cali...)   │  ───────►   │  (Bogotá, Medellín...)│
  └──────────────────────┘             └──────────────────────┘
```

In [ ]:
%%sql
-- 1. Unión (∪): Todas las ciudades donde hay clientes O departamentos (Sin duplicados)
-- Expresión: π_{ciudad}(clientes) ∪ π_{ciudad}(departamentos)
SELECT ciudad FROM clientes
UNION
SELECT ciudad FROM departamentos;

-- 2. Intersección (∩): Ciudades donde HAY TANTO clientes COMO departamentos
-- Expresión: π_{ciudad}(clientes) ∩ π_{ciudad}(departamentos)
SELECT ciudad FROM clientes
INTERSECT
SELECT ciudad FROM departamentos;

-- 3. Diferencia (-): Ciudades que tienen clientes PERO NO departamentos
-- Expresión: π_{ciudad}(clientes) - π_{ciudad}(departamentos)
SELECT ciudad FROM clientes
EXCEPT
SELECT ciudad FROM departamentos;

## 4. Producto Cartesiano ($\times$) y Theta Join ($\bowtie$)
- **Producto Cartesiano ($R \times S$):** Combina cada fila de $R$ con cada fila de $S$. Produce $|R| \times |S|$ filas.
- **Theta Join ($R \bowtie_{\theta} S$):** Es un Producto Cartesiano seguido de una Selección con predicado $\theta$.
$$\text{empleados} \bowtie_{\text{empleados.depto\_id} = \text{departamentos.depto\_id}} \text{departamentos}$$

In [ ]:
%%sql
-- 1. Producto Cartesiano (×): 10 empleados × 5 departamentos = 50 filas
SELECT e.nombre AS empleado, d.nombre AS departamento
FROM empleados e
CROSS JOIN departamentos d
LIMIT 5;

-- 2. Theta Join (⋈): Filtra solo las parejas que coinciden en depto_id
SELECT e.nombre AS empleado, e.salario, d.nombre AS departamento, d.ciudad
FROM empleados e
JOIN departamentos d ON e.depto_id = d.depto_id;

## 5. Retos Prácticos de Álgebra Relacional 🏋️‍♂️
### ✏️ Reto 1: Selección y Proyección Combinada
Expresa en notación formal y escribe la consulta SQL para obtener los nombres y presupuestos de los departamentos ubicados en **'Bogotá'** con presupuesto $\ge 100.000.000$.
### ✏️ Reto 2: Operación de Conjuntos (Unión)
Escribe la consulta SQL para obtener una lista unificada de todos los nombres de personas/entidades registradas en el sistema (nombres de `empleados` y nombres de `clientes`).
### ✏️ Reto 3: Diferencia de Conjuntos
Escribe la consulta SQL para encontrar los nombres de clientes que son de categoría **'Empresarial'** PERO QUE NO están radicados en **'Bogotá'**.

In [ ]:
-- 🛠 SOLUCIÓN OFICIAL DE LOS RETOS - MÓDULO 00

-- Solución Reto 1: π_{nombre, presupuesto}(σ_{ciudad = 'Bogotá' ∧ presupuesto >= 100000000}(departamentos))
SELECT nombre, presupuesto
FROM departamentos
WHERE ciudad = 'Bogotá' AND presupuesto >= 100000000.00;

-- Solución Reto 2: π_{nombre}(empleados) ∪ π_{nombre}(clientes)
SELECT nombre FROM empleados
UNION
SELECT nombre FROM clientes;

-- Solución Reto 3: π_{nombre}(σ_{categoria_cliente = 'Empresarial'}(clientes)) - π_{nombre}(σ_{ciudad = 'Bogotá'}(clientes))
SELECT nombre FROM clientes WHERE categoria_cliente = 'Empresarial'
EXCEPT
SELECT nombre FROM clientes WHERE ciudad = 'Bogotá';